#  Forward pass

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы: 
* Deep Learning with PyTorch (2020) Авторы: Eli Stevens, Luca Antiga, Thomas Viehmann 
* https://pytorch.org/docs/stable/generated/torch.matmul.html
* https://machinelearningmastery.com/choose-an-activation-function-for-deep-learning/
* https://machinelearningmastery.com/loss-and-loss-functions-for-training-deep-learning-neural-networks/
* https://kidger.site/thoughts/jaxtyping/
* https://github.com/patrick-kidger/torchtyping/tree/master

## Задачи для совместного разбора

In [2]:
from torchtyping import TensorType, patch_typeguard
from typeguard import typechecked
import torch as th

Scalar = TensorType[()]
patch_typeguard()

1\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте нейрон с заданными весами `weights` и `bias`. Пропустите вектор `inputs` через нейрон и выведите результат. 

In [3]:
class Neuron:
    def __init__(self, n_features: int, bias: float):
        # <создать атрибуты объекта weights и bias>
        self.weights: TensorType[n_features] = th.rand(n_features)
        self.bias: float = bias

    def forward(self, inputs: TensorType["n_features"]) -> Scalar:
        return inputs*self.weights + self.bias


In [4]:
import torch as th
inputs = th.tensor([1.0, 2.0, 3.0, 4.0])

In [5]:
neuron = Neuron(4, True)
neuron.forward(inputs)

tensor([1.0635, 2.3877, 3.5651, 1.5270])

2\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию активации ReLU:

![](https://wikimedia.org/api/rest_v1/media/math/render/svg/f4353f4e3e484130504049599d2e7b040793e1eb)

Создайте матрицу размера (4,3), заполненную числами из стандартного нормального распределения, и проверьте работоспособность функции активации.

In [15]:
class ReLU:
    @typechecked
    def forward(self, inputs: TensorType["n_features"]) -> TensorType["n_features"]:
        mask = inputs >= 0
        outputs = inputs.clone() 
        outputs[~mask] = 0
        return outputs

3\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию потерь MSE:

![](https://wikimedia.org/api/rest_v1/media/math/render/svg/e258221518869aa1c6561bb75b99476c4734108e)
где $Y_i$ - правильный ответ для примера $i$, $\hat{Y_i}$ - предсказание модели для примера $i$, $n$ - количество примеров в батче.

In [31]:
class MSELoss:
    @typechecked
    def forward(self, y_pred: TensorType["batch"], y_true: TensorType["batch"]) -> Scalar:
        return th.mean(th.pow((y_pred  - y_true), 2))

In [32]:
y_pred = th.tensor([1.0, 3.0, 5.0])
y_true = th.tensor([2.0, 3.0, 4.0])
MSELoss().forward(y_pred, y_true)

tensor(0.6667)

## Задачи для самостоятельного решения

### Cоздание полносвязных слоев

<p class="task" id="1"></p>

1\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте полносвязный слой из `n_neurons` нейронов с `n_features` весами у каждого нейрона (инициализируются из стандартного нормального распределения) и опциональным вектором смещения. 

$$y = xW^T + b$$

Пропустите вектор `inputs` через слой и выведите результат. Результатом прогона сквозь слой должна быть матрица размера `batch_size` x `n_neurons`.

- [ ] Проверено на семинаре

In [45]:
class Linear:
    def __init__(self, n_neurons: int, n_features: int, bias: bool = False) -> None:
        self.weights = th.rand((n_neurons, n_features))
        self.is_bias = bias
        if self.is_bias:
            self.bias = th.zeros(n_neurons)

    def forward(self, inputs: TensorType["batch", "feats"]) -> TensorType["batch", "n_neurons"]:
        vector = inputs@self.weights.T 
        if self.is_bias:
            return vector + self.bias
        return vector

<p class="task" id="2"></p>

2\. Используя решение предыдущей задачи, создайте 2 полносвязных слоя и пропустите тензор `inputs` последовательно через эти два слоя. Количество нейронов в первом слое выберите произвольно, количество нейронов во втором слое выберите так, чтобы результатом прогона являлась матрица `batch_size x 7`. 

- [ ] Проверено на семинаре

In [54]:
layer_1 = Linear(10, 7)
layer_2 = Linear(7, 10)

inputs = th.rand((20, 7))
out_1 = layer_1.forward(inputs)
out_2 = layer_2.forward(out_1)
out_2.shape

torch.Size([20, 7])

### Создание функций активации

<p class="task" id="3"></p>

3\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию активации softmax:

![](https://wikimedia.org/api/rest_v1/media/math/render/svg/6d7500d980c313da83e4117da701bf7c8f1982f5)

$$\overrightarrow{x} = (x_1, ..., x_J)$$

Создайте матрицу размера (4,3), заполненную числами из стандартного нормального распределения, и проверьте работоспособность функции активации. Строки матрицы трактовать как выходы линейного слоя некоторого классификатора для 4 различных примеров. Функция должна применяться переданной на вход матрице построчно.

- [ ] Проверено на семинаре

In [116]:
class Softmax:
    def forward(self, inputs: TensorType["batch", "feats"]) -> TensorType["batch", "feats"]:
        exps = th.exp(inputs)
        return exps / exps.sum(dim=1, keepdim=True)

In [80]:
t = th.randn(4, 3)

out = Softmax().forward(t)
out

tensor([[0.3287, 0.4893, 0.1820],
        [0.7853, 0.1599, 0.0548],
        [0.0293, 0.5424, 0.4283],
        [0.1404, 0.4383, 0.4213]])

In [81]:
out.sum(dim=1)

tensor([1.0000, 1.0000, 1.0000, 1.0000])

<p class="task" id="4"></p>

4 Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию активации ELU:

![](https://wikimedia.org/api/rest_v1/media/math/render/svg/eb23becd37c3602c4838e53f532163279192e4fd)

Создайте матрицу размера 4x3, заполненную числами из стандартного нормального распределения, и проверьте работоспособность функции активации.

- [ ] Проверено на семинаре

In [94]:
class ELU:
    def __init__(self, alpha: float) -> None:
        self.alpha = alpha
        pass
    
    def forward(self, inputs: TensorType["batch", "feats"]) -> TensorType["batch", "feats"]:
        mask = inputs < 0
        outs = inputs.clone()
        outs[mask] = self.alpha * (outs[mask] - 1)
        return outs

In [111]:
t = th.randn(4, 3)
t

tensor([[-0.8227, -0.4781, -2.4824],
        [-1.5962, -0.4485, -0.6640],
        [-0.3368, -0.5344,  1.4722],
        [-0.3348,  0.1581, -0.0854]])

In [112]:
elu = ELU(0.05)
elu.forward(t)

tensor([[-0.0911, -0.0739, -0.1741],
        [-0.1298, -0.0724, -0.0832],
        [-0.0668, -0.0767,  1.4722],
        [-0.0667,  0.1581, -0.0543]])

### Создание функции потерь

<p class="task" id="5"></p>

5 Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию потерь CrossEntropyLoss:

$$y_i = (y_{i,1},...,y_{i,k})$$ 

<img src="https://i.ibb.co/93gy1dN/Screenshot-9.png" width="200">

$$ CrossEntropyLoss = \frac{1}{n}\sum_{i=1}^{n}{L_i}$$
где $y_i$ - вектор правильных ответов для примера $i$, $\hat{y_i}$ - вектор предсказаний модели для примера $i$; $k$ - количество классов, $n$ - количество примеров в батче.

Создайте полносвязный слой с 2 нейронами и прогнать через него батч `inputs`. Полученный результат пропустите через функцию активации Softmax. Посчитайте значение функции потерь, трактуя вектор `y` как вектор правильных ответов.

- [ ] Проверено на семинаре

In [125]:
class CrossEntropyLoss:
    def forward(self, y_pred: TensorType["batch", float], y_true: TensorType["batch", int]):
        batch_size = y_pred.shape[0]
        correct_probs = y_pred[th.arange(batch_size), y_true]
        return -th.log(correct_probs).mean()

In [126]:
probs, y

(tensor([[0.5298, 0.4702],
         [0.6164, 0.3836],
         [0.4154, 0.5846],
         [0.5361, 0.4639]]),
 tensor([0, 1, 1, 0]))

In [128]:
inputs = th.randn(4, 3)
y = th.tensor([0, 1, 1, 0])

linear = Linear(n_neurons=2, n_features=3, bias=True)
logits = linear.forward(inputs)

softmax = Softmax()
probs = softmax.forward(logits)

criterion = CrossEntropyLoss()
loss = criterion.forward(probs, y)

print(probs)
loss

tensor([[0.3959, 0.6041],
        [0.5880, 0.4120],
        [0.5034, 0.4966],
        [0.6834, 0.3166]])


tensor(0.7235)

<p class="task" id="6"></p>

6 Модифицируйте MSE, добавив L2-регуляризацию.

$$MSE_R = MSE + \lambda\sum_{i=1}^{m}w_i^2$$

где $\lambda$ - коэффициент регуляризации; $w_i$ - веса модели.

- [ ] Проверено на семинаре

In [129]:
class MSERegularized:
    def __init__(self, lambda_: float) -> None:
        self.lambda_ = lambda_

    def data_loss(
            self, 
            y_pred: TensorType["batch"], 
            y_true: TensorType["batch"],
    ) -> Scalar:
        return th.mean(th.pow((y_pred  - y_true), 2))

    def reg_loss(self, weights: TensorType["batch", 1])  -> Scalar:
        return th.pow(weights, 2).sum()

    def forward(
        self, 
        y_pred: TensorType["batch"], 
        y_true: TensorType["batch"], 
        weights: TensorType["batch", 1],
    ) -> Scalar:
        return self.data_loss(y_pred, y_true) + lambda_ * self.reg_loss(weights)